In [ ]:
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import DummyVecEnv, VecTransposeImage
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback
from gymnasium.wrappers import RecordVideo
import torch
import os
import numpy as np
import ale_py
import time

In [ ]:
print("=== DQN Training for ALE/Assault-v5 ===")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

# Create save directory
save_dir = "assault_dqn_model"
os.makedirs(save_dir, exist_ok=True)

def make_assault_env():
    """Create Assault environment with proper Atari wrappers"""
    env = gym.make("ALE/Assault-v5", render_mode="rgb_array")
    env = Monitor(env)
    env = AtariWrapper(env)  # Standard Atari preprocessing
    return env

# Training callback to track progress
class ProgressCallback(BaseCallback):
    def __init__(self, check_freq: int, save_path: str):
        super().__init__()
        self.check_freq = check_freq
        self.save_path = save_path
        self.best_mean_reward = -np.inf

    def _on_step(self) -> bool:
        if self.n_calls % self.check_freq == 0:
            # Get recent episode rewards
            if hasattr(self.training_env, 'get_attr'):
                try:
                    episode_rewards = self.training_env.get_attr('episode_returns')[0]
                    if len(episode_rewards) > 10:
                        mean_reward = np.mean(episode_rewards[-10:])
                        print(f"Step {self.n_calls}: Mean reward (last 10 episodes): {mean_reward:.2f}")
                        
                        if mean_reward > self.best_mean_reward:
                            self.best_mean_reward = mean_reward
                            self.model.save(os.path.join(self.save_path, "best_assault_model"))
                            print(f"New best model saved! Reward: {mean_reward:.2f}")
                except:
                    pass
        return True

# Create vectorized environment
def create_env():
    return make_assault_env()

train_env = DummyVecEnv([create_env])
train_env = VecTransposeImage(train_env)  # For CNN processing

# Set device and hyperparameters
device = "cuda" if torch.cuda.is_available() else "cpu"
buffer_size = 100000 if device == "cuda" else 50000
total_timesteps = 500000 if device == "cuda" else 100000

print(f"Device: {device}")
print(f"Buffer size: {buffer_size}")
print(f"Training timesteps: {total_timesteps}")

# Create DQN model
model = DQN(
    "CnnPolicy",
    train_env,
    buffer_size=buffer_size,
    learning_starts=50000,
    batch_size=32,
    target_update_interval=10000,
    gamma=0.99,
    exploration_fraction=0.1,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.01,
    learning_rate=1e-4,
    train_freq=4,
    gradient_steps=1,
    verbose=1,
    device=device,
    tensorboard_log=None  # Disable to avoid issues
)

print("\nStarting DQN training on Assault...")
print("This will take several minutes. Watch the reward improve over time!")

# Create callback
callback = ProgressCallback(check_freq=10000, save_path=save_dir)

# Train the model
try:
    model.learn(
        total_timesteps=total_timesteps,
        callback=callback,
        log_interval=10,
        progress_bar=False
    )
    print("\nTraining completed successfully!")
except KeyboardInterrupt:
    print("\nTraining interrupted by user")

# Save final model
final_path = os.path.join(save_dir, "final_assault_model")
model.save(final_path)
print(f"Final model saved to: {final_path}")

# Clean up training environment
train_env.close()

print("\n=== Training Complete! ===")
print("Ready for demonstration and video recording.")

In [ ]:
print("=== Video Recording for Presentation ===")

# Configuration
save_dir = "assault_dqn_model"
video_dir = "presentation_videos"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Create video directory
os.makedirs(video_dir, exist_ok=True)

# Load the trained model
model_path = None
for model_name in ["best_assault_model", "final_assault_model"]:
    potential_path = os.path.join(save_dir, model_name)
    if os.path.exists(potential_path + ".zip"):
        model_path = potential_path
        print(f"Using model: {model_name}")
        break

if model_path is None:
    print("ERROR: No trained model found! Please run the training cell first.")
    exit()

def make_video_env():
    """Create Assault environment with video recording"""
    # Base environment
    env = gym.make("ALE/Assault-v5", render_mode="rgb_array")
    env = Monitor(env)
    
    # Add video recording wrapper
    env = RecordVideo(
        env,
        video_folder=video_dir,
        episode_trigger=lambda episode: True,  # Record all episodes
        name_prefix="dqn_assault_gameplay"
    )
    
    # Apply Atari wrappers after video recording
    env = AtariWrapper(env)
    return env

def create_video_env():
    return make_video_env()

# Create environment for recording
print("Setting up video recording environment...")
video_env = DummyVecEnv([create_video_env])
video_env = VecTransposeImage(video_env)

# Load model
print(f"Loading trained model from: {model_path}")
model = DQN.load(model_path, env=video_env, device=device)

print("\n RECORDING GAMEPLAY VIDEOS")
print("=" * 40)
print("Recording multiple episodes for presentation...")

# Record multiple episodes
num_episodes = 3
episode_stats = []

for episode in range(num_episodes):
    print(f"\nRecording Episode {episode + 1}/{num_episodes}")
    
    obs = video_env.reset()
    episode_reward = 0
    step_count = 0
    
    while step_count < 3000:  # Allow longer episodes for better videos
        # Get action from trained agent
        action, _states = model.predict(obs, deterministic=True)
        
        # Execute action
        obs, reward, done, info = video_env.step(action)
        
        episode_reward += reward[0] if hasattr(reward, '__len__') else reward
        step_count += 1
        
        # End if game over
        if done[0] if hasattr(done, '__len__') else done:
            break
    
    episode_stats.append({
        'episode': episode + 1,
        'reward': episode_reward,
        'steps': step_count
    })
    
    print(f"   Episode {episode + 1} recorded: Score = {episode_reward:.0f}, Steps = {step_count}")

# Close environment to finalize videos
video_env.close()

print("\n RECORDING SUMMARY")
print("=" * 30)
for stats in episode_stats:
    print(f"Episode {stats['episode']}: Score {stats['reward']:.0f}, Steps {stats['steps']}")

avg_reward = np.mean([s['reward'] for s in episode_stats])
print(f"\nAverage Score: {avg_reward:.1f}")

print("\n VIDEO FILES CREATED")
print("=" * 25)
video_files = []
for file in os.listdir(video_dir):
    if file.endswith('.mp4'):
        video_files.append(file)
        print(f" {file}")

if video_files:
    print(f"\n SUCCESS! {len(video_files)} video(s) saved to: {video_dir}/")
    print("\n FOR YOUR PRESENTATION:")
    print("1. Use these videos to show DQN in action")
    print("2. Explain how the agent learned through trial and error")
    print("3. Point out intelligent behaviors:")
    print("   - Aiming at enemies")
    print("   - Dodging enemy fire")
    print("   - Maximizing score")
    print("4. Compare with random play to show learning")
    
    print("\n VIDEO USAGE TIPS:")
    print("- Play videos at normal speed to show real-time performance")
    print("- Pause to explain specific decisions the agent makes")
    print("- Highlight the difference from early (random) vs trained play")
else:
    print(" No video files found. Check for errors above.")

print("\nPRESENTATION MATERIALS READY!")
print("Your DQN Assault demonstration is complete.")